In [ ]:
# from utils import *
import random 
import numpy as np 
import keras 
import tensorflow as tf 
from keras import layers , ops
from keras.utils import image_dataset_from_directory


img_path = r"C:\Users\blade_mx4\Documents\Datasets\DOG_CAT"
print(" ================= Import Done =============ds====  === ")

 ================= Import Done =============ds====  === 


In [ ]:

def load_img_path(path):
    ds = image_dataset_from_directory(
        path ,
        image_size=(128,128) ,
        batch_size=8 ,
        label_mode= "int" ,
        shuffle=False 
    )
    image = np.array(ds.file_paths) 
    labels = np.concatenate([y.numpy() for _,y in ds]) 
    return image ,labels

def load_and_decode(paired_path , labels):
    def read_img_path(file_path) :
        img  = tf.io.read_file(file_path)
        img = tf.image.decode_jpeg(img , channels=3)
        img = tf.image.resize(img,(224,224)) 
        img = img / 127.5
        return img 
    img_1 = read_img_path(paired_path[0])
    img_2 = read_img_path(paired_path[1])
    return (img_1 , img_2),labels


def make_pairs (x,y) : 
    num_classes = max(y) + 1
    digit_indices = [np.where(y == i)[0] for i in range(num_classes)]
    pairs = []
    labels = []
    for idx1 in range(len(x)):
        # add a matching example
        x1 = x[idx1]
        label1 = y[idx1]
        idx2 = random.choice(digit_indices[label1])
        x2 = x[idx2]
        pairs += [[x1, x2]]
        labels += [0]
        # add a non-matching example
        label2 = random.randint(0, num_classes - 1)
        while label2 == label1:
            label2 = random.randint(0, num_classes - 1)

        idx2 = random.choice(digit_indices[label2])
        x2 = x[idx2]

        pairs += [[x1, x2]]
        labels += [1]

    return np.array(pairs), np.array(labels).astype("float32")

def euclidean_distance(vects):
    x, y = vects
    sum_square = ops.sum(ops.square(x - y), axis=1, keepdims=True)
    return ops.sqrt(ops.maximum(sum_square, keras.backend.epsilon()))



img_ds_path , label = load_img_path(img_path)
img_pair , label_pair = make_pairs(img_ds_path , label)

ds = tf.data.Dataset.from_tensor_slices((img_pair , label_pair)) 
ds = ds.map(
    load_and_decode ,num_parallel_calls=tf.data.AUTOTUNE 
)
ds = ds.batch(8)
ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)



Found 720 files belonging to 2 classes.


In [ ]:

def build_embedding_model():
    Input = layers.Input((224, 224, 3))
    # x = layers.Rescaling(scale=1./127.5, offset=-1)(Input)
    x = layers.BatchNormalization()(Input)
    
    x = layers.Conv2D(16, kernel_size=(3, 3), strides=(1, 1), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, kernel_size=(3, 3), strides=(1, 1), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(64, kernel_size=(3, 3), strides=(1, 1), activation='relu')(x)
    x = layers.Conv2D(128, kernel_size=(3, 3), strides=(1, 1), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.BatchNormalization()(x)

    x = layers.AveragePooling2D(pool_size=(2, 2))(x)
    x = layers.Dropout(0.2)(x)          # <-- fixed: now reassigned
    x = layers.Flatten()(x)

    # x = layers.BatchNormalization()(x)

    x = layers.Dense(64, activation='relu')(x)   # embedding, pre-normalization
    # x = layers.UnitNormalization()(x)            # <-- L2-normalize so distances live in [0, 2]

    return keras.Model(Input, x)

build_embedding_model().summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 224, 224, 3)         │              12 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 222, 222, 16)        │             448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 111, 111, 16)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 109, 109, 32)        │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 54, 54, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 54, 54, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 52, 52, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 50, 50, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 25, 25, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 25, 25, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ average_pooling2d (AveragePooling2D) │ (None, 12, 12, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 12, 12, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 18432)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │       1,179,712 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,277,804 (4.87 MB)

 Trainable params: 1,277,478 (4.87 MB)

 Non-trainable params: 326 (1.27 KB)

In [ ]:
Embedding = build_embedding_model()
Input_layer_A = layers.Input((224,224,3))
Input_layer_B = layers.Input((224,224,3)) 

Vector_A = Embedding(Input_layer_A)
Vector_B = Embedding(Input_layer_B)

distance = layers.Concatenate()([Vector_A , Vector_B])
dense = layers.Dense(64,activation='relu')(distance) 
output = layers.Dense(1,activation = 'sigmoid')(dense)
# distance = layers.Lambda(euclidean_distance)([Vector_A , Vector_B])

model = keras.Model(inputs =[Input_layer_A ,Input_layer_B ] , outputs = output)
model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)    │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_3 (InputLayer)    │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ functional_1 (Functional)     │ (None, 64)                │       1,277,804 │ input_layer_2[0][0],       │
│                               │                           │                 │ input_layer_3[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate (Concatenate)     │ (None, 128)               │               0 │ functional_1[0][0],        │
│                               │                           │                 │ functional_1[1][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 64)                │           8,256 │ concatenate[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_3 (Dense)               │ (None, 1)                 │              65 │ dense_2[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,286,125 (4.91 MB)

 Trainable params: 1,285,799 (4.90 MB)

 Non-trainable params: 326 (1.27 KB)

In [ ]:
# def Loss(y_true, y_pred, margin=1):
#     """
#     Contrastive loss.
#     y_true = 0 -> similar pair   -> minimize distance
#     y_true = 1 -> dissimilar pair -> push distance >= margin
#     """
#     squr_pred = ops.square(y_pred)
#     margin_sqr = ops.square(ops.maximum(margin - y_pred, 0))
#     LOSS = ops.mean((1 - y_true) * squr_pred + y_true * margin_sqr)
#     return LOSS


# def accuracy(y_true, y_pred):
#     y_true = ops.cast(y_true, "float32")

#     y_true = ops.reshape(y_true, [-1])
#     y_pred = ops.reshape(y_pred, [-1])

#     # 0 for similar (<0.5), 1 for dissimilar (>0.5)
#     preds = ops.cast(y_pred > 0.5, "float32")
#     return ops.mean(ops.equal(y_true, preds))


In [ ]:
model.compile(
    loss = 'binary_crossentropy', 
    optimizer=keras.optimizers.Adam(learning_rate =0.001) ,
    metrics = ['accuracy'] ,
    jit_compile= True 

)

model.fit(
    ds , 
    epochs = 100 ,
    verbose = 1 ,

)
model.save("siamese.keras")